In [ ]:
import dsautils.calstatus as cs
from dsautils.dsa_store import DsaStore
from astropy.time import Time
import time
import datetime
import yaml
from dsacalib.weights import average_beamformer_solutions
import glob
import os
import numpy as np
from pkg_resources import resource_filename
import astropy.units as u
from dsautils import cnf
from dsacalib.plotting import summary_plot, plot_current_beamformer_solutions
from dsacalib.plotting import plot_beamformer_weights
from dsacalib.routines import get_files_for_cal, calibrate_measurement_set
from dsacalib.weights import get_good_solution, write_beamformer_solutions
from dsacalib.ms_io import convert_calibrator_pass_to_ms, uvh5_to_ms
import matplotlib.pyplot as plt
%matplotlib inline
from matplotlib.backends.backend_pdf import PdfPages
import h5py
myconf = cnf.Conf()
ETCD = DsaStore()

# Calibration resources that may be useful

## Manual Calibration - if calibration pipeline fails

The "calsources" file contains a list of sources, from which you can select one to analyze. I suggest making your own "calsources" file in the same format and pulling sources from there.  

In [ ]:
# The calibrator pass you want
date = '2024-02-07'
calname = '1459+716'
dec = '+071p6'
# Parameters that don't need to be changed
antennas = [v for k, v in myconf.get('corr')['antenna_order'].items()]
duration = 10*u.min 
calsources = resource_filename(
    'dsacalib',
    f'data/calibrator_sources_dec{dec}.csv'
)
refcorr = 'corr03'
filelength = 5*u.min
msdir = '/operations/calibration/'
hdf5dir = '/operations/correlator/'
date_specifier = '{0}*'.format(date)
msname = '{0}/{1}_{2}'.format(msdir, date, calname)



In [ ]:
# Get a list of the files for each calibrator
filenames = get_files_for_cal(
    calsources,
    hdf5dir,
    'sb01',
    duration,
    filelength,
    date_specifier
)
#print(filenames)
print(filenames[date][calname])

In [ ]:
files = sorted(glob.glob(
    '/operations/correlator/{0}[{1}{2}{3}]???_sb??.hdf5'.format(
        filenames[date][calname]['files'][-1][:-4],
        int(filenames[date][calname]['files'][-1][-4])-1,
        filenames[date][calname]['files'][-1][-4],
        int(filenames[date][calname]['files'][-1][-4])+1
    )
), key = lambda x: x[-7:-5])

print(len(files),files)
assert len(files) < 17

## Create measurement set within notebook

In [ ]:
import dsacalib.config as configuration
config = configuration.Configuration()
print(config)

In [ ]:
cal = filenames[date][calname]['cal']
convert_calibrator_pass_to_ms(cal,date,filenames[date][calname]['files'],msdir=msdir,hdf5dir=hdf5dir,refmjd=config.refmjd)